# LeNet5 PYNQ Overlay Test

`lenet5.bit`, `lenet5.hwh`, `image.txt`, `weight.txt`, `bias.txt`가 같은 Jupyter 폴더에 있을 때 바로 실행하는 테스트용 노트북입니다.

기대 결과: `pred = 9`, scores = `[-128, -116, -128, -27, 20, -101, -128, -128, -41, 127]`.


In [1]:
from pynq import Overlay, allocate, MMIO
import numpy as np
import time, os, gc

BIT_FILE = "lenet5.bit"

for f in ["lenet5.bit", "lenet5.hwh", "image.txt", "weight.txt", "bias.txt"]:
    print(f, "OK" if os.path.exists(f) else "MISSING")

ol = Overlay(BIT_FILE)
print("Overlay loaded")
print("IP names:")
for k in ol.ip_dict.keys():
    print("  ", k)

npu_key = next(k for k in ol.ip_dict.keys() if "lenet5_axi_wrapper" in k)
dma_key = next(k for k in ol.ip_dict.keys() if "axi_dma" in k)

npu_info = ol.ip_dict[npu_key]
npu = MMIO(npu_info["phys_addr"], npu_info["addr_range"])

try:
    dma = getattr(ol, dma_key)
except AttributeError:
    dma = getattr(ol, dma_key.split("/")[-1])

print("NPU:", npu_key, hex(npu_info["phys_addr"]), npu_info["addr_range"])
print("DMA:", dma_key)


lenet5.bit OK
lenet5.hwh OK
image.txt OK
weight.txt OK
bias.txt OK


Overlay loaded
IP names:
   lenet5_axi_wrapper_0
   axi_dma_0
   processing_system7_0
NPU: lenet5_axi_wrapper_0 0x40000000 4096
DMA: axi_dma_0


In [2]:
# AXI-Lite register map
REG_CONTROL       = 0x00
REG_STATUS        = 0x04
REG_LOAD_TYPE     = 0x08
REG_CURRENT_STATE = 0x0C
REG_PREDICTED     = 0x10
REG_ACTIVE_LOAD   = 0x14
REG_SCORE0        = 0x20

CTRL_LOAD_BEGIN   = 0x1
CTRL_START        = 0x2
CTRL_PS_READ_DONE = 0x4

LOAD_IMAGE  = 0
LOAD_WEIGHT = 1
LOAD_BIAS   = 2

STATUS_LOAD_ACTIVE  = 0
STATUS_LOAD_DONE    = 1
STATUS_BUSY         = 2
STATUS_DONE         = 3
STATUS_RESULT_VALID = 4
STATUS_TREADY       = 5
STATUS_IDLE         = 6
STATUS_FINISH       = 7

def signed32(x):
    x = int(x) & 0xFFFFFFFF
    return x - 0x100000000 if x & 0x80000000 else x

def read_status():
    return int(npu.read(REG_STATUS))

def print_status(prefix="STATUS"):
    s = read_status()
    print(f"{prefix}=0x{s:08X} active={(s>>0)&1} load_done={(s>>1)&1} busy={(s>>2)&1} "
          f"done={(s>>3)&1} result_valid={(s>>4)&1} tready={(s>>5)&1} idle={(s>>6)&1} finish={(s>>7)&1} "
          f"state={npu.read(REG_CURRENT_STATE)}")
    return s

def wait_predicate(pred, label, timeout_s=10.0, poll_s=0.001):
    t0 = time.time()
    last = None
    while True:
        s = read_status()
        last = s
        if pred(s):
            return s
        if time.time() - t0 > timeout_s:
            print_status("TIMEOUT_STATUS")
            raise TimeoutError(f"Timeout waiting for {label}, last STATUS=0x{last:08X}")
        time.sleep(poll_s)

def read_hex_lines_u8(path):
    vals = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if s:
                vals.append(int(s, 16) & 0xFF)
    return vals

def read_hex_lines_u32(path):
    vals = []
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if s:
                vals.append(int(s, 16) & 0xFFFFFFFF)
    return vals

def pack4_u8_to_u32(values):
    values = np.asarray(values, dtype=np.uint8)
    assert values.size % 4 == 0, f"u8 line count must be multiple of 4, got {values.size}"
    v = values.astype(np.uint32)
    words = (v[0::4] | (v[1::4] << 8) | (v[2::4] << 16) | (v[3::4] << 24)).astype(np.uint32)
    return words

_live_buffers = []

def dma_send_words(words, label):
    words = np.asarray(words, dtype=np.uint32)
    buf = allocate(shape=(len(words),), dtype=np.uint32)
    buf[:] = words
    buf.flush()
    print(f"DMA send {label}: {len(words)} words, {len(words)*4} bytes")
    dma.sendchannel.transfer(buf)
    dma.sendchannel.wait()
    _live_buffers.append(buf)
    return buf

def begin_load(load_type):
    npu.write(REG_LOAD_TYPE, int(load_type))
    npu.write(REG_CONTROL, CTRL_LOAD_BEGIN)
    wait_predicate(lambda s: (s & (1 << STATUS_LOAD_ACTIVE)) and not (s & (1 << STATUS_LOAD_DONE)),
                   "load_active asserted", timeout_s=2.0)
    wait_predicate(lambda s: (s & (1 << STATUS_TREADY)), "s_axis_tready", timeout_s=2.0)

def load_to_npu(load_type, words, label):
    print("\n--- Load", label, "---")
    begin_load(load_type)
    print_status("BEFORE_DMA")
    buf = dma_send_words(words, label)
    wait_predicate(lambda s: (s & (1 << STATUS_LOAD_DONE)), f"{label} load_done", timeout_s=10.0)
    print_status("AFTER_DMA")
    return buf

def release_to_idle():
    npu.write(REG_CONTROL, CTRL_PS_READ_DONE)
    wait_predicate(lambda s: (s & (1 << STATUS_IDLE)), "core idle", timeout_s=2.0)
    print_status("IDLE")


In [3]:
image_u8  = read_hex_lines_u8("image.txt")
weight_u8 = read_hex_lines_u8("weight.txt")
bias_u32  = read_hex_lines_u32("bias.txt")

print("line counts:")
print(" image ", len(image_u8),  "expected 8192")
print(" weight", len(weight_u8), "expected 61560")
print(" bias  ", len(bias_u32),  "expected 1736")

assert len(image_u8) == 8192
assert len(weight_u8) == 61560
assert len(bias_u32) == 1736

image_words  = pack4_u8_to_u32(image_u8)
weight_words = pack4_u8_to_u32(weight_u8)
bias_words   = np.asarray(bias_u32, dtype=np.uint32)

print("DMA word counts:")
print(" image_words ", len(image_words),  "expected 2048")
print(" weight_words", len(weight_words), "expected 15390")
print(" bias_words  ", len(bias_words),   "expected 1736")

print_status("INITIAL")


line counts:
 image  8192 expected 8192
 weight 61560 expected 61560
 bias   1736 expected 1736
DMA word counts:
 image_words  2048 expected 2048
 weight_words 15390 expected 15390
 bias_words   1736 expected 1736
INITIAL=0x00000040 active=0 load_done=0 busy=0 done=0 result_valid=0 tready=0 idle=1 finish=0 state=0


64

In [4]:
# 전체 로드 + 추론 실행
image_buf  = load_to_npu(LOAD_IMAGE,  image_words,  "image")
weight_buf = load_to_npu(LOAD_WEIGHT, weight_words, "weight")
bias_buf   = load_to_npu(LOAD_BIAS,   bias_words,   "bias")

print("\n--- Start inference ---")
npu.write(REG_CONTROL, CTRL_START)
wait_predicate(lambda s: (s & (1 << STATUS_RESULT_VALID)), "result_valid", timeout_s=10.0)
print_status("RESULT")

pred = int(npu.read(REG_PREDICTED)) & 0xF
scores = [signed32(npu.read(REG_SCORE0 + 4*i)) for i in range(10)]
argmax = int(np.argmax(scores))

print("scores:", scores)
print("pred  :", pred)
print("argmax:", argmax)

expected_scores = [-128, -116, -128, -27, 20, -101, -128, -128, -41, 127]
expected_pred = 9

print("expected scores:", expected_scores)
print("expected pred  :", expected_pred)

if scores == expected_scores and pred == expected_pred and argmax == expected_pred:
    print("✅ PASS: hardware result matches expected simulation result")
else:
    print("⚠️ Result differs from expected. Check file versions, packing, or hardware state.")

release_to_idle()



--- Load image ---
BEFORE_DMA=0x00000061 active=1 load_done=0 busy=0 done=0 result_valid=0 tready=1 idle=1 finish=0 state=0
DMA send image: 2048 words, 8192 bytes
AFTER_DMA=0x00000042 active=0 load_done=1 busy=0 done=0 result_valid=0 tready=0 idle=1 finish=0 state=0

--- Load weight ---
BEFORE_DMA=0x00000061 active=1 load_done=0 busy=0 done=0 result_valid=0 tready=1 idle=1 finish=0 state=0
DMA send weight: 15390 words, 61560 bytes
AFTER_DMA=0x00000042 active=0 load_done=1 busy=0 done=0 result_valid=0 tready=0 idle=1 finish=0 state=0

--- Load bias ---
BEFORE_DMA=0x00000061 active=1 load_done=0 busy=0 done=0 result_valid=0 tready=1 idle=1 finish=0 state=0
DMA send bias: 1736 words, 6944 bytes
AFTER_DMA=0x00000042 active=0 load_done=1 busy=0 done=0 result_valid=0 tready=0 idle=1 finish=0 state=0

--- Start inference ---
RESULT=0x0000009A active=0 load_done=1 busy=0 done=1 result_valid=1 tready=0 idle=0 finish=1 state=6
scores: [-128, -116, -128, -27, 20, -101, -128, -128, -41, 127]
pred